# Plot gallery

A tour of the plots in `arcsine_laws.plots`, built on the samplers in `arcsine_laws.walks`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from arcsine_laws import (
    brownian_refinements,
    donsker,
    gaussian,
    rademacher,
    random_walk,
)
from arcsine_laws.plots import plot_histogram, plot_paths, plot_refinements

rng = np.random.default_rng(1729)

## A simple random walk

Flip a fair coin, step up on heads and down on tails.

In [ ]:
walk = random_walk(200, rademacher(), rng=rng)

fig, ax = plt.subplots(figsize=(14, 4))
plot_paths(ax, walk)
ax.set_title("One simple random walk, 200 steps")

## Many walks

A fan of 200 walks, with one picked out. Where the fan is darker, more walks pass through.

In [ ]:
walks = random_walk(1000, rademacher(), n_paths=200, rng=rng)

fig, ax = plt.subplots(figsize=(14, 4))
plot_paths(ax, walks, highlight=0)
ax.set_title("200 simple random walks, 1000 steps each")

## Zooming out: Donsker scaling

Squash n steps into the interval [0, 1] and shrink space by sqrt(n). As n grows the walk looks more and more like Brownian motion.

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(14, 8), sharex=True, sharey=True)
for ax, n_steps in zip(axs, [16, 256, 4096]):
    walk = random_walk(n_steps, rademacher(), n_paths=5, rng=rng)
    plot_paths(ax, donsker(walk), alpha=0.8)
    ax.set_title(f"{n_steps} steps")
axs[0].set_xlabel("")
axs[1].set_xlabel("")
fig.tight_layout()

## Brownian motion, exactly on a grid

With Gaussian steps of variance dt, a random walk *is* Brownian motion observed every dt.

In [ ]:
n_steps = 1024
bm = random_walk(n_steps, gaussian(variance=1/n_steps), n_paths=200, dt=1/n_steps, rng=rng)

fig, ax = plt.subplots(figsize=(14, 4))
plot_paths(ax, bm, highlight=0)
ax.set_title("200 Brownian motion paths on [0, 1]")

## Brownian motion by midpoint refinement

Start from B(0) = 0 and a random B(1). Repeatedly fill in each midpoint from the Brownian bridge between its neighbours. Each level keeps every point of the one before.

In [ ]:
refinements = brownian_refinements(10, rng=rng)

fig, ax = plt.subplots(figsize=(14, 4))
plot_refinements(ax, refinements[:6])
ax.set_title("The first five midpoint refinements")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
plot_refinements(ax, refinements[::2])
ax.set_title("Every other level, down to 1024 steps")

## The central limit theorem

The endpoint of a Donsker scaled walk is approximately N(0, 1). Each bar is one possible endpoint; with few steps there are only a handful.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, n_steps in zip(axs, [4, 16, 256]):
    walks = donsker(random_walk(n_steps, rademacher(), n_paths=20_000, rng=rng))
    plot_histogram(ax, walks.x[:, -1], normal=(0, 1))
    ax.set_title(f"Endpoint after {n_steps} steps")
fig.tight_layout()